In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import Fullscreen

# Load GeoJSON with ZIP shapes
zip_geo = gpd.read_file("../Dashboard_Final/simplified.geojson")

# Load ZIP-level metrics
zip_counts = pd.read_csv("../Dashboard_final/food_programs.csv", sep="|", dtype={"zipcode": str})
zip_counts = zip_counts["zipcode"].value_counts().reset_index()
zip_counts.columns = ["ZIP", "program_count"]

gap_metrics = pd.read_csv("../Dashboard_Final/avg_gap_scores.csv")
gap_metrics = gap_metrics[["zipcode", "gap_score", "raw_demand", "population"]]
gap_metrics["zipcode"] = gap_metrics["zipcode"].astype(str)

# Merge metrics with shapefile
combined = zip_geo.merge(zip_counts, on="ZIP", how="left")
combined = combined.merge(gap_metrics, left_on="ZIP", right_on="zipcode", how="left")
combined = combined.fillna(0)

# --- Choose metric to visualize ---
metric = "gap_score"  # Change to 'program_count', 'raw_demand', 'population' as needed
metric_pretty = {
    "gap_score": "Gap Score",
    "program_count": "Program Count",
    "raw_demand": "Raw Demand",
    "population": "Population"
}[metric]

# --- Create Map ---
m = folium.Map(location=[29.4241, -98.4936], zoom_start=9)
Fullscreen().add_to(m)

folium.Choropleth(
    geo_data=combined,
    data=combined,
    columns=["ZIP", metric],
    key_on="feature.properties.ZIP",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name=f"{metric_pretty} by ZIP Code",
    nan_fill_color="white"
).add_to(m)

folium.GeoJson(
    combined,
    style_function=lambda feature: {
        "fillOpacity": 0,
        "color": "black",
        "weight": 1.2
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["ZIP", metric],
        aliases=["ZIP Code:", metric_pretty]
    )
).add_to(m)

m
